In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 19. The label-noise ceiling

This turns the strongest qualitative finding in the project into a number.

The error analysis found that 58 of 140 missed hate posts are political criticism and 26 are
posts you would not call hate on reading. Dr Hunter's reaction on 22 October was
*"that's something you can comment on, that about half fall into those two categories"*. A
comment is worth some credit. **A measured ceiling is worth more**, because it converts an
opinion about the corpus into a quantity that explains the results.

## The idea

If a competent reader of romanised Hinglish disagrees with the corpus label on a share of
posts, then no classifier can agree with those labels more often than a reader does and still
be doing something sensible. The disagreement rate is therefore an estimate of label noise,
and label noise puts a ceiling on achievable accuracy.

## Why the sample has to come from all four cells

The earlier annotation only covered missed hate posts, which is a biased sample: those are
exactly the posts where the model and the label disagree, so disagreement is guaranteed to
look high. `four_cell_inspection.xlsx` from notebook 16 samples all four cells, which makes
the estimate correctable: measure agreement inside each cell, then reweight by how large each
cell actually is in the full test set.

**Fill in the `your_verdict` column first.** Use `hate`, `not hate`, or `unsure`. This
notebook does nothing until that column has values.

In [2]:
import os, sys
from pathlib import Path
import numpy as np, pandas as pd

# Pointing to your Google Drive dissertation folder
ROOT = Path('/content/drive/MyDrive/dissertation')
if not ROOT.exists():
    # Fallback to handle spelling variations
    ROOT = Path('/content/drive/MyDrive/disseratation')
if not ROOT.exists():
    ROOT = Path.cwd().parent

W = ROOT / 'writing'
FIG = W / 'figures'; FIG.mkdir(parents=True, exist_ok=True)

XL  = W / 'four_cell_inspection.xlsx'
CSV = W / 'four_cell_inspection.csv'

if XL.exists():
    ann = pd.read_excel(XL, sheet_name='all')
    src = XL.name
elif CSV.exists():
    ann = pd.read_csv(CSV); src = CSV.name
else:
    raise FileNotFoundError(f"Could not find 'four_cell_inspection.xlsx' or 'four_cell_inspection.csv' inside '{W}'. Please make sure you have run notebook 16 and that your project files are in Google Drive.")

ann['your_verdict'] = ann['your_verdict'].astype(str).str.strip().str.lower()
VALID = {'hate': 1, 'not hate': 0, 'nothate': 0, 'not-hate': 0, 'no': 0, 'yes': 1}
done = ann[ann['your_verdict'].isin(VALID)].copy()
unsure = ann[ann['your_verdict'].isin(['unsure', 'maybe', '?'])]

print(f'{src}: {len(ann)} sampled posts')
print(f'  annotated:  {len(done)}')
print(f'  marked unsure: {len(unsure)}')
print(f'  not yet done:  {len(ann) - len(done) - len(unsure)}')

if len(done) < 20:
    print("\nFill in more of the your_verdict column before reading anything below."
          "\nAbout 60 posts, one model, is enough for a usable estimate.")

four_cell_inspection.xlsx: 480 sampled posts
  annotated:  0
  marked unsure: 0
  not yet done:  480

Fill in more of the your_verdict column before reading anything below.
About 60 posts, one model, is enough for a usable estimate.


In [3]:
if len(done) >= 20:
    done['my_label'] = done['your_verdict'].map(VALID)
    done['agrees']   = (done['my_label'] == done['label']).astype(int)

    per_cell = (done[done.split == 'within']
                .groupby('cell')
                .agg(n_read=('agrees', 'size'),
                     agreement=('agrees', 'mean'))
                .round(3).reset_index())
    display(per_cell)
    print('\nagreement = how often your reading matched the corpus label, by cell')

## Reweighting to the whole test set

Each cell was sampled to roughly equal size, but the cells are not equally large in the real
test set. Weighting each cell's agreement rate by its true share gives an estimate for the
test set as a whole rather than for the sample.

In [4]:
if len(done) >= 20:
    cc = pd.read_csv(W / 'confusion_counts.csv')
    MODEL_FOR_WEIGHTS = 'TF-IDF + LogReg'
    row = cc[(cc.model == MODEL_FOR_WEIGHTS) & (cc.split == 'within')].iloc[0]
    sizes = {'TN correct not-hate': row['true_not'], 'FP false alarm': row['false_hate'],
             'FN missed hate': row['missed_hate'], 'TP caught hate': row['true_hate']}
    total = sum(sizes.values())
    print(f'cell sizes in the real test set ({MODEL_FOR_WEIGHTS}): {sizes}, total {total}')

    pc = per_cell.set_index('cell')
    covered = [c for c in sizes if c in pc.index]
    if len(covered) < 4:
        print(f'\nOnly {len(covered)} of 4 cells annotated. Read some from the others '
              'before trusting the weighted figure.')

    w = np.array([sizes[c] for c in covered], dtype=float); w /= w.sum()
    a = np.array([pc.loc[c, 'agreement'] for c in covered], dtype=float)
    weighted = float((w * a).sum())
    unweighted = float(done[done.split == 'within']['agrees'].mean())

    print(f'\nunweighted agreement over the sample : {unweighted:.3f}')
    print(f'cell-size weighted agreement          : {weighted:.3f}')
    print(f'implied label-noise rate              : {1 - weighted:.3f}')
    print(f'\nBest macro-F1 any model could reach against these labels, roughly: {weighted:.3f}')

    obs = {'TF-IDF + LogReg': 0.642, 'XLM-RoBERTa': 0.653,
           'MuRIL': 0.654, 'IndicBERT v1': 0.652}
    print('\nagainst observed macro-F1:')
    for m, v in obs.items():
        print(f'  {m:18s} {v:.3f}   ({100*v/weighted:.0f}% of the estimated ceiling)')

In [5]:
if len(done) >= 20:
    out = {'n_annotated': len(done),
           'n_unsure': len(unsure),
           'unweighted_agreement': round(unweighted, 3),
           'weighted_agreement': round(weighted, 3),
           'implied_label_noise': round(1 - weighted, 3)}
    pd.DataFrame([out]).to_csv(W / 'label_noise_ceiling.csv', index=False)
    per_cell.to_csv(W / 'agreement_by_cell.csv', index=False)
    print('saved label_noise_ceiling.csv and agreement_by_cell.csv')

## How to write this up

One paragraph, and it is the paragraph that explains the whole Results chapter:

> A native reader of romanised Hinglish re-annotated a stratified sample of N test posts drawn
> from all four cells of the confusion matrix. Weighting each cell by its size in the full
> test set, the reader agreed with the corpus label on X% of posts. Since no classifier can
> exceed a competent human reader's agreement with a label set and still be judged sensible,
> this places an approximate ceiling of X on accuracy against these labels. The best model
> reached Y, which is Z% of that ceiling.

**State the limitations honestly.** It is a single annotator, which means no inter-annotator
agreement statistic, the sample is modest, and the annotator is not blind to the model's
prediction, since it is in the spreadsheet. The last of those is the easiest to fix if you
have time: hide the `pred` and `p_hate` columns before reading. Say all three in the text.
Hunter is far more likely to reward a limitation you named than one he found.